# AIST-FYP Colab: Build Sentence Index

This notebook pre-builds sentence-level embedding indexes for claim-to-evidence retrieval.

## What this notebook does
1. Mounts Google Drive and clones the repo
2. Installs project dependencies
3. Validates benchmark data paths for the selected dataset
4. Builds sentence index artifacts with scripts/build_sentence_index.py
5. Verifies output files and persists them to Drive

## Dataset Prerequisites

Before running this notebook, ensure:

- `benchmark/RAGTruth/dataset` exists for `DATASET="ragtruth"`
- `benchmark/CiteEval` oracle data exists for `DATASET="citeeval"`
- `data/` is linked to your Drive artifact root

Output folder defaults to `data/indexes/{dataset}_sentences/{split}` unless overridden.

## 🔑 Setup API Keys (Colab Secrets)

To run the evaluation, you likely need API keys for the LLM providers (DeepSeek, OpenAI, etc.). Colab provides a secure way to store these using the **Secrets** feature.

1. Click on the **Key icon** (Secrets) in the left sidebar.
2. Add a new secret with the name:
   - `OPENAI_API_KEY`: Your OpenAI API key.
   - `DEEPSEEK_API_KEY`: Your DeepSeek API key.
   - `HUGGINGFACE_TOKEN`: (Optional) For gated models.
3. Make sure to toggle the **"Notebook access"** switch to **ON** for this notebook.

The next cell will automatically load these secrets into the environment variables used by the project scripts.

In [ ]:
# ==============================
# Setup API Keys (Secrets)
# ==============================
try:
    from google.colab import userdata
    import os

    # Define keys to load
    secret_keys = ["OPENAI_API_KEY", "DEEPSEEK_API_KEY", "HUGGINGFACE_TOKEN"]
    loaded_any = False

    for key in secret_keys:
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
                print(f"✅ Loaded secret: {key}")
                loaded_any = True
        except Exception:
            # Silently skip if not found or no access
            pass
    
    if not loaded_any:
        print("ℹ️ No secrets loaded. If you need API keys, add them via the 🔑 (Secrets) tab.")
except ImportError:
    print("⚠️ 'google.colab.userdata' not found. If running locally, please export your API keys manually.")

### Configuration (Smoke Test Settings)
This notebook defaults to a **smoke test** for both RAGTruth and CiteEval datasets. 

**For full evaluation:**
1. In the cell below, change `RAGTRUTH_MAX_SAMPLES = 10` and `CITEEVAL_MAX_SAMPLES = 10` to `None`.
2. Run the notebook.

In [ ]:
# ==============================
# Parameters (edit this cell)
# ==============================
REPO_URL = "https://github.com/xiashuidaolaoshuren/AIST-FYP.git"
REPO_BRANCH = "main"
REPO_DIR = "/content/AIST-FYP"
COLAB_ENV_PROJECT = "colab/env"
COLAB_UV_EXTRAS = ["evaluation"]
INSTALL_SPACY_MODEL = True

DATASET = "ragtruth"  # ragtruth | citeeval
SPLIT = "test"        # test | train | all
ORACLE_DATASET = "asqa"  # asqa | eli5 | msmarco (citeeval only)
ENCODER_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
DEVICE = "cuda"
BATCH_SIZE = 32
OUTPUT_DIR_OVERRIDE = ""

ARTIFACTS_IN_PROJECT = True
DRIVE_DATA_ROOT = "/content/drive/MyDrive/data"
DRIVE_RAGTRUTH_DATASET_ROOT = "/content/drive/MyDrive/AIST-FYP/benchmark/RAGTruth/dataset"
DRIVE_CITEEVAL_ROOT = "/content/drive/MyDrive/AIST-FYP/benchmark/CiteEval"

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/AIST-FYP-colab-outputs"
DRIVE_WORK_ROOT = f"{DRIVE_OUTPUT_DIR}/work_eval"
LOCAL_WORK_ROOT = DRIVE_WORK_ROOT
RUN_TAG = "colab_build_sentence_index"

In [ ]:
import os
import json
import shutil
import subprocess
import sys
from pathlib import Path
from datetime import datetime

def _normalized_unique_paths(entries):
    ordered = []
    seen = set()
    for entry in entries:
        if not entry:
            continue
        resolved = str(Path(entry).resolve())
        if resolved not in seen:
            ordered.append(resolved)
            seen.add(resolved)
    return ordered

def run(cmd, cwd=None, check=True, stream=True):
    env = os.environ.copy()
    repo_path = str(Path(cwd).resolve()) if cwd else str(Path(os.getcwd()).resolve())

    existing_pp = env.get('PYTHONPATH', '').split(os.pathsep)
    pp_entries = _normalized_unique_paths([repo_path, *existing_pp])

    site_pkgs = [p for p in sys.path if 'site-packages' in p]
    pp_entries = _normalized_unique_paths([*pp_entries, *site_pkgs])

    env['PYTHONPATH'] = os.pathsep.join(pp_entries)

    print(f"\n$ {cmd}")

    if stream:
        process = subprocess.Popen(
            cmd,
            shell=True,
            cwd=cwd,
            env=env,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1,
            executable='/bin/bash',
        )
        out_lines = []
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            out_lines.append(line)
        process.wait()
        completed = subprocess.CompletedProcess(
            args=cmd,
            returncode=process.returncode,
            stdout="".join(out_lines),
            stderr=None,
        )
    else:
        completed = subprocess.run(
            cmd,
            shell=True,
            cwd=cwd,
            env=env,
            text=True,
            capture_output=True,
            executable='/bin/bash',
        )
        if completed.stdout:
            print(completed.stdout)

    if completed.returncode != 0:
        if not stream and completed.stderr:
            print(completed.stderr)
        if check:
            raise RuntimeError(f"Command failed ({completed.returncode}): {cmd}")
    return completed

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

def ensure_symlink_dir(link_path: Path, target_path: Path):
    target_path = Path(target_path)
    link_path = Path(link_path)
    ensure_dir(target_path)
    ensure_dir(link_path.parent)

    if link_path.is_symlink():
        current_target = Path(os.readlink(link_path))
        if current_target == target_path:
            return
        link_path.unlink()
    elif link_path.exists():
        if link_path.is_dir():
            shutil.rmtree(link_path)
        else:
            link_path.unlink()

    os.symlink(target_path, link_path, target_is_directory=True)

def copytree_merge(src, dst):
    src_p = Path(src)
    dst_p = Path(dst)
    if not src_p.exists():
        return
    for p in src_p.rglob('*'):
        rel = p.relative_to(src_p)
        t = dst_p / rel
        if p.is_dir():
            t.mkdir(parents=True, exist_ok=True)
        else:
            t.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, t)

def exists_or_raise(path, msg):
    if not Path(path).exists():
        raise FileNotFoundError(f"{msg}: {path}")

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

if not str(LOCAL_WORK_ROOT).startswith('/content/drive/'):
    print(f"⚠️ LOCAL_WORK_ROOT is not on Drive: {LOCAL_WORK_ROOT}")
    print("Artifacts may not survive Colab runtime reset.")

ensure_dir(LOCAL_WORK_ROOT)
print("Persistent LOCAL_WORK_ROOT:", LOCAL_WORK_ROOT)

# Clone/update repo
if Path(REPO_DIR).exists() and (Path(REPO_DIR) / '.git').exists():
    print(f"Repo dir already exists: {REPO_DIR}")
    run("git fetch --all", cwd=REPO_DIR, check=False)
    run(f"git checkout {REPO_BRANCH}", cwd=REPO_DIR, check=False)
    run(f"git pull origin {REPO_BRANCH}", cwd=REPO_DIR, check=False)
else:
    run(f"git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")

run("git rev-parse --abbrev-ref HEAD", cwd=REPO_DIR)
run("git log -1 --oneline", cwd=REPO_DIR)

# Ensure repo root is available first for script imports (src.*).
repo_root = str(Path(REPO_DIR))
existing_pp = [p for p in os.environ.get('PYTHONPATH', '').split(os.pathsep) if p]
ordered_pp = []
for entry in [repo_root, *existing_pp]:
    if entry not in ordered_pp:
        ordered_pp.append(entry)
os.environ['PYTHONPATH'] = os.pathsep.join(ordered_pp)
print('PYTHONPATH (repo-first):', os.environ['PYTHONPATH'])
print('src/utils/config.py exists:', (Path(REPO_DIR) / 'src' / 'utils' / 'config.py').exists())

# Link project data path to Drive data artifacts root
drive_data_root = Path(DRIVE_DATA_ROOT)
if not drive_data_root.exists():
    raise FileNotFoundError(
        f"Drive data root not found: {drive_data_root}. Place artifacts under this path before running evaluation."
    )

project_data_path = Path(REPO_DIR) / 'data'
ensure_symlink_dir(project_data_path, drive_data_root)
resolved_data_path = project_data_path.resolve()
print(f"Data artifact path: {project_data_path} -> {resolved_data_path}")

# Link RAGTruth dataset path to Drive if available
drive_ragtruth_dataset = Path(DRIVE_RAGTRUTH_DATASET_ROOT)
project_ragtruth_dataset = Path(REPO_DIR) / 'benchmark' / 'RAGTruth' / 'dataset'
if drive_ragtruth_dataset.exists():
    ensure_symlink_dir(project_ragtruth_dataset, drive_ragtruth_dataset)
    resolved_ragtruth_dataset = project_ragtruth_dataset.resolve()
    print(f"RAGTruth dataset path: {project_ragtruth_dataset} -> {resolved_ragtruth_dataset}")
elif RUN_RAGTRUTH:
    print(
        f"⚠️ Drive RAGTruth dataset not found at {drive_ragtruth_dataset}. "
        "If your dataset is elsewhere, update DRIVE_RAGTRUTH_DATASET_ROOT in the parameters cell."
    )

# Link CiteEval/CiteBench benchmark folder to Drive if available
drive_citeeval_root = Path(DRIVE_CITEEVAL_ROOT)
project_citeeval_root = Path(REPO_DIR) / 'benchmark' / 'CiteEval'
if drive_citeeval_root.exists():
    ensure_symlink_dir(project_citeeval_root, drive_citeeval_root)
    resolved_citeeval_root = project_citeeval_root.resolve()
    print(f"CiteEval benchmark path: {project_citeeval_root} -> {resolved_citeeval_root}")
elif RUN_CITEEVAL or RUN_CITEEVAL_METRIC or RUN_CITEEVAL_VERIFIER_MODULE_EVAL:
    print(
        f"⚠️ Drive CiteEval root not found at {drive_citeeval_root}. "
        "If your benchmark is elsewhere, update DRIVE_CITEEVAL_ROOT in the parameters cell."
    )

# Persist runtime outputs via symlink to Drive-backed workspace
symlink_map = {
    Path(REPO_DIR) / 'outputs' / 'ragtruth_eval': Path(RAGTRUTH_OUTPUT_DIR),
    Path(REPO_DIR) / 'outputs' / 'mitigation_eval': Path(VERIFIER_RAG_OUTPUT_DIR),
    Path(REPO_DIR) / 'outputs' / 'mitigation_eval_citebench': Path(VERIFIER_CITE_OUTPUT_DIR),
    Path(REPO_DIR) / 'benchmark' / 'CiteEval' / 'data' / 'system_eval_outputs': Path(CITEEVAL_SYSTEM_OUTPUTS_DIR),
    Path(REPO_DIR) / 'benchmark' / 'CiteEval' / 'data' / 'metric_eval_outputs': Path(CITEEVAL_METRIC_OUTPUTS_DIR),
    Path(REPO_DIR) / 'benchmark' / 'CiteEval' / 'data' / 'tmp' / 'sampling': Path(CITEEVAL_TMP_SAMPLING_DIR),
}

for link_path, target_path in symlink_map.items():
    ensure_symlink_dir(link_path, target_path)
    print(f"Persisted path: {link_path} -> {target_path}")

In [ ]:
# Install dependencies
run("python -m pip install -U pip wheel setuptools", stream=True)
run("python -m pip install -U faiss-cpu rank_bm25", stream=True)
run("python -m pip install -U uv", stream=True)

uv_project = Path(REPO_DIR) / COLAB_ENV_PROJECT
extras_args = " ".join(f"--extra {extra}" for extra in COLAB_UV_EXTRAS)
sync_cmd = f"uv sync --project {uv_project} {extras_args}"
result = run(sync_cmd, cwd=REPO_DIR, check=False, stream=True)

spacy_model_wheel = "https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl"
spacy_install_cmd = "python -m spacy download en_core_web_sm"

if result.returncode == 0:
    uv_python = uv_project / ".venv" / "bin" / "python"
    os.environ["PATH"] = f"{uv_python.parent}:{os.environ.get('PATH', '')}"
    spacy_install_cmd = f"uv pip install --python {uv_python} {spacy_model_wheel}"
    run(f"{uv_python} - <<\"PY\"\nimport sys\nprint('python executable:', sys.executable)\nPY", cwd=REPO_DIR)
    print(f"✅ uv sync complete: {uv_project}")
else:
    print('\n⚠️ uv sync failed. Falling back to pip requirements install...')
    requirements_path = Path(REPO_DIR) / 'requirements.txt'
    pytorch_index = 'https://download.pytorch.org/whl/cu121'
    install_cmd = f"pip install --extra-index-url {pytorch_index} -r {requirements_path}"
    fallback_result = run(install_cmd, cwd=REPO_DIR, check=False, stream=True)

    if fallback_result.returncode != 0:
        print('\n⚠️ Full requirements install failed. Falling back to Colab-torch-compatible install...')
        filtered = []
        skip_prefixes = ('torch==', 'torchvision==', 'torchaudio==')
        for raw in requirements_path.read_text(encoding='utf-8').splitlines():
            line = raw.strip()
            if not line or line.startswith('#'):
                continue
            if any(line.startswith(prefix) for prefix in skip_prefixes):
                continue
            filtered.append(line)

        temp_req = Path(REPO_DIR) / 'requirements.colab.filtered.txt'
        temp_req.write_text('\n'.join(filtered) + '\n', encoding='utf-8')
        run(f"pip install -r {temp_req}", cwd=REPO_DIR, stream=True)

# spaCy model required by verifier
if INSTALL_SPACY_MODEL:
    run(spacy_install_cmd, cwd=REPO_DIR, stream=True)

In [ ]:
# Runtime and API key setup
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Set keys in Colab before running CiteEval modules that need them:
# os.environ['DEEPSEEK_API_KEY'] = '...'
# os.environ['OPENAI_API_KEY'] = '...'

# Provider defaults from parameter cell
os.environ['CITEEVAL_PROVIDER'] = CITEEVAL_PROVIDER
os.environ['CITEEVAL_ROOT'] = str(Path(REPO_DIR) / 'benchmark/CiteEval')

repo_root = str(Path(REPO_DIR).resolve())

# Critical: keep only repo root in global PYTHONPATH to avoid src-package shadowing.
# CiteEval scripts are executed from repo context and should not globally override src.*
existing_pp = [p for p in os.environ.get('PYTHONPATH', '').split(os.pathsep) if p]
ordered_pp = []
seen = set()
for entry in [repo_root, *existing_pp]:
    resolved = str(Path(entry).resolve())
    if resolved not in seen:
        ordered_pp.append(resolved)
        seen.add(resolved)
os.environ['PYTHONPATH'] = os.pathsep.join(ordered_pp)

print('PYTHONPATH (top 6):', ordered_pp[:6])
print('CITEEVAL_PROVIDER =', os.environ.get('CITEEVAL_PROVIDER'))
print('CITEEVAL_MODEL_NAME =', CITEEVAL_MODEL_NAME if CITEEVAL_MODEL_NAME else '(script default)')
print('DEEPSEEK_API_KEY set =', bool(os.environ.get('DEEPSEEK_API_KEY')))
print('OPENAI_API_KEY set =', bool(os.environ.get('OPENAI_API_KEY')))

if os.environ['PYTHONPATH'].split(os.pathsep)[0] != repo_root:
    raise RuntimeError('PYTHONPATH ordering error: repo root is not first entry.')

In [ ]:
# Validate required benchmark paths for selected dataset
repo = Path(REPO_DIR)
ragtruth_dataset = repo / 'benchmark/RAGTruth/dataset'
citeeval_root = repo / 'benchmark/CiteEval'

if DATASET == 'ragtruth':
    exists_or_raise(ragtruth_dataset, 'Missing RAGTruth dataset directory')
elif DATASET == 'citeeval':
    exists_or_raise(citeeval_root, 'Missing CiteEval benchmark directory')
else:
    raise ValueError(f"Unsupported DATASET={DATASET}. Expected 'ragtruth' or 'citeeval'.")

print('Preflight path checks passed for DATASET=', DATASET)

In [ ]:
# Validate required paths for selected strategy
repo = Path(REPO_DIR)
faiss_index = repo / f"data/indexes/{STRATEGY}/faiss.index"
index_meta = repo / f"data/indexes/{STRATEGY}/metadata.pkl"
chunks_file = repo / f"data/processed/wiki_chunks_{STRATEGY}.jsonl"
ragtruth_dataset = repo / 'benchmark/RAGTruth/dataset'
citeeval_root = repo / 'benchmark/CiteEval'

exists_or_raise(faiss_index, 'Missing FAISS index')
exists_or_raise(index_meta, 'Missing index metadata')
exists_or_raise(chunks_file, 'Missing processed chunks')

if RUN_RAGTRUTH:
    exists_or_raise(ragtruth_dataset, 'Missing RAGTruth dataset directory')

if RUN_CITEEVAL or RUN_CITEEVAL_METRIC or RUN_CITEEVAL_VERIFIER_MODULE_EVAL:
    exists_or_raise(citeeval_root, 'Missing CiteEval benchmark directory')

print('Preflight path checks passed.')

In [ ]:
# Resolve output directory and build command
repo = Path(REPO_DIR).resolve()
split_value = SPLIT

if OUTPUT_DIR_OVERRIDE.strip():
    output_dir = OUTPUT_DIR_OVERRIDE.strip()
else:
    output_dir = f"data/indexes/{DATASET}_sentences/{split_value}"

if DATASET == 'ragtruth':
    build_cmd = (
        f'python scripts/build_sentence_index.py '
        f'--dataset ragtruth --split {split_value} '
        f'--encoder "{ENCODER_MODEL}" --device {DEVICE} --batch-size {BATCH_SIZE} '
        f'--output-dir {output_dir}'
    )
else:
    build_cmd = (
        f'python scripts/build_sentence_index.py '
        f'--dataset citeeval --split {split_value} --oracle-dataset {ORACLE_DATASET} '
        f'--encoder "{ENCODER_MODEL}" --device {DEVICE} --batch-size {BATCH_SIZE} '
        f'--output-dir {output_dir}'
    )

print('Build command:')
print(build_cmd)
print('Output dir:', output_dir)

In [ ]:
# Run sentence index build
run(build_cmd, cwd=str(repo))
print('Sentence index build completed.')

In [ ]:
# Verify output files and mirror a copy manifest to Drive
out_path = (Path(REPO_DIR) / output_dir).resolve()
required_files = [
    out_path / 'sentences.jsonl',
    out_path / 'embeddings.npy',
    out_path / 'sample_index.json',
]
for file_path in required_files:
    exists_or_raise(file_path, 'Missing sentence-index artifact')
    print('Found:', file_path)

mirror_dir = Path(DRIVE_OUTPUT_DIR) / 'sentence_indexes' / DATASET / split_value
ensure_dir(mirror_dir)
for file_path in required_files:
    shutil.copy2(file_path, mirror_dir / file_path.name)

print('Mirrored artifacts to:', mirror_dir)